# Obtaining Geographic Coordinates for NYC Taxi Zones

## Objective

The original NYC Taxi trip dataset provides pickup locations using `PULocationID`, which identifies predefined Taxi and Limousine Commission (TLC) zones. However, clustering algorithms such as DBSCAN and geospatial visualization libraries such as Folium require latitude and longitude coordinates.

To obtain accurate spatial coordinates, official NYC Taxi Zone geographic boundaries are used. The centroid (geometric center) of each taxi zone polygon is calculated and used as the representative latitude and longitude for that zone.

This approach is preferred over geocoding libraries because it ensures consistency with TLC zone definitions, eliminates external API dependencies, and provides reproducible results suitable for spatial machine learning tasks.

## Import Required Libraries

GeoPandas is used to handle geographic data and compute centroid coordinates for NYC Taxi Zones.

In [1]:
import geopandas as gpd
import pandas as pd

## Load NYC Taxi Zone Boundaries

The official NYC Taxi Zone shapefile contains polygon geometries representing each TLC taxi zone.

Important columns include:

- `LocationID` : Unique Taxi Zone identifier
- `zone` : Zone name
- `borough` : Borough name
- `geometry` : Polygon boundary

In [10]:
zones = gpd.read_file(
    "/home/ed/Desktop/NYC_Taxi_Demand_Forecasting/USECASE3_Pickup_hotspot_Detection/data/mapsfile/taxi_zones.shp"
)

zones.head()

,OBJECTID,Shape_Leng,Shape_Area,zone,LocationID,borough,geometry
0,1,0.116357,0.000782,Newark Airport,1,EWR,"POLYGON ((933100.918 192536.086, 933091.011 19..."
1,2,0.433470,0.004866,Jamaica Bay,2,Queens,"MULTIPOLYGON (((1033269.244 172126.008, 103343..."
2,3,0.084341,0.000314,Allerton/Pelham Gardens,3,Bronx,"POLYGON ((1026308.77 256767.698, 1026495.593 2..."
3,4,0.043567,0.000112,Alphabet City,4,Manhattan,"POLYGON ((992073.467 203714.076, 992068.667 20..."
4,5,0.092146,0.000498,Arden Heights,5,Staten Island,"POLYGON ((935843.31 144283.336, 936046.565 144..."


## Verify Coordinate Reference System

Centroid calculations should be performed using a projected coordinate system to ensure geographic accuracy.

In [11]:
print(zones.crs)

EPSG:2263


## Convert Geometry to WGS84 Coordinate System

The WGS84 coordinate system (EPSG:4326) is used because Folium and most mapping libraries require latitude and longitude values.

In [12]:
zones = zones.to_crs(epsg=4326)

## Calculate Taxi Zone Centroids

The centroid represents the geometric center of each taxi zone polygon and serves as the approximate latitude and longitude of that zone.

These centroid coordinates will later be used for:

- DBSCAN hotspot detection
- Interactive hotspot visualization
- Driver positioning recommendations

In [13]:
zones["latitude"] = (
    zones.geometry.centroid.y
)

zones["longitude"] = (
    zones.geometry.centroid.x
)

/tmp/ipykernel_4548/4190867913.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  zones.geometry.centroid.y
/tmp/ipykernel_4548/4190867913.py:6: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  zones.geometry.centroid.x



## Extract Relevant Geographic Information

Only the required columns are retained for subsequent merging operations.

In [14]:
zone_coordinates = zones[
    [
        "LocationID",
        "zone",
        "borough",
        "latitude",
        "longitude"
    ]
]

zone_coordinates.head()

,LocationID,zone,borough,latitude,longitude
0,1,Newark Airport,EWR,40.691831,-74.174000
1,2,Jamaica Bay,Queens,40.616745,-73.831299
2,3,Allerton/Pelham Gardens,Bronx,40.864474,-73.847422
3,4,Alphabet City,Manhattan,40.723752,-73.976968
4,5,Arden Heights,Staten Island,40.552659,-74.188484


## Merge Geographic Coordinates with Pickup Demand Data

The hotspot dataset containing pickup demand information is merged with the centroid coordinates using `PULocationID`.

This creates a unified dataset containing both demand intensity and geographic location.

In [15]:
hotspot_df = pd.read_csv(
    "../data/zone_hotspot_data.csv"
)

hotspot_df = hotspot_df.merge(
    zone_coordinates,
    left_on="PULocationID",
    right_on="LocationID",
    how="left"
)

hotspot_df.head()

,PULocationID,Zone,Borough,pickup_count,demand_percentage,hotspot_level,LocationID,zone,borough,latitude,longitude
0,237,Upper East Side South,Manhattan,160343,4.311502,Very High,237,Upper East Side South,Manhattan,40.768615,-73.965635
1,236,Upper East Side North,Manhattan,153640,4.131264,Very High,236,Upper East Side North,Manhattan,40.780436,-73.957012
2,132,JFK Airport,Queens,152589,4.103003,Very High,132,JFK Airport,Queens,40.646985,-73.786533
3,161,Midtown Center,Manhattan,146641,3.943066,Very High,161,Midtown Center,Manhattan,40.758028,-73.977698
4,186,Penn Station/Madison Sq West,Manhattan,110700,2.976639,Very High,186,Penn Station/Madison Sq West,Manhattan,40.748497,-73.992438


## Validate Coordinate Availability

All taxi zones should successfully map to valid centroid coordinates.

In [16]:
hotspot_df[
    [
        "latitude",
        "longitude"
    ]
].isnull().sum()

latitude     0
longitude    0
dtype: int64

## Save Dataset for Downstream Tasks

The enriched dataset is saved for use in:

- DBSCAN Hotspot Detection
- Hotspot Visualization
- Driver Recommendation System

In [17]:
hotspot_df.to_csv(
    "../data/zone_hotspot_coordinates.csv",
    index=False
)

print(
    "Dataset saved successfully."
)

Dataset saved successfully.


# Conclusion

This notebook enriched the hotspot dataset with accurate geographic coordinates derived from official NYC Taxi Zone boundaries.

Instead of relying on external geocoding services, zone centroid coordinates were calculated directly from TLC-defined polygon geometries. This approach ensures consistency, reproducibility, and spatial accuracy required for geospatial machine learning tasks.

The resulting dataset provides a robust foundation for hotspot detection using DBSCAN, interactive hotspot visualization using Folium, and downstream driver recommendation systems aimed at improving operational efficiency within New York City's taxi ecosystem.